# **Linear Regression and Regularization with Mathematical Insights: PySpark ML on Diabetes Data**

### **`Dr Amin Karami (PhD, FHEA, EE), UEL UK - Docklands Campus`**

`E: amin.karami@ymail.com`

`W: https://www.youtube.com/@AminKarami`

`W: https://www.aminkarami.com`

---

**Learning Outcomes**:

`Master Linear Regression and Regularization`: Understand and apply linear regression, Lasso, and Ridge techniques to predict diabetes outcomes using PySpark ML.

`Data Handling Expertise`: Acquire skills in preprocessing, feature selection, and data management within a Jupyter Notebook environment for large-scale datasets.

`Model Optimization Strategies`: Learn to fine-tune machine learning models through regularization to enhance predictive accuracy and model robustness.

`Accuracy Assessment`: Gain proficiency in evaluating model performance with various accuracy metrics, ensuring reliable predictive insights.


In [ ]:
!pip3 install pyspark

# **Step 1:** Import the required libraries and initialize SparkSession.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("LinearRegressionExample") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()

spark

# **Step 2:** Load and preprocess the data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data Description

Diabetes Data: https://drive.google.com/file/d/1JgxAN-lTKXbrsaHtetOpybk4XTLRYqCF/view?usp=sharing (204 MB)

We have a dataset of diabetes patients, including ten health measurement features and one variable indicating disease progression after one year.

| Feature 	| Description                           	|
|---------	|---------------------------------------	|
| age     	| Age in years                          	|
| sex     	| Sex                                   	|
| bmi     	| Body Mass Index                       	|
| bp      	| Average blood pressure                	|
| tc      	| T-Cells (a type of white blood cells) 	|
| ldl     	| low-density lipoproteins              	|
| hdl     	| high-density lipoproteins             	|
| tch     	| thyroid stimulating hormone           	|
| ltg     	| lamotrigine                           	|
| glu     	| blood sugar level                     	|

In [ ]:
# Load the data from a CSV file
df = spark.read.csv("/content/drive/MyDrive/Files/Diabetes.csv", header=True, inferSchema=True)

# get familiar with data
df.show()

# more info
print(df.count())
print(df.rdd.getNumPartitions())

+---+---+------+-----+------+------+-----+-----+----+-----+------+-----------+
| id|age|   sex|  bmi|    bp|    tc|  ldl|  hdl| tch|  ltg|   glu|progression|
+---+---+------+-----+------+------+-----+-----+----+-----+------+-----------+
|  0| 22|  Male|20.33|106.25|140.89|-0.45|62.83|0.77|-0.54|130.98|      29.83|
|  1| 41|Female|25.99|132.23|129.64|-1.11|37.26|0.81| 1.64|151.19|      46.66|
|  2| 51|Female|32.76| 127.0|220.36|-1.69|49.56|0.41|-0.88|176.22|      59.97|
|  3| 26|  Male|35.87| 138.4|194.19|-0.04|55.57|0.45|-1.38|125.32|      42.44|
|  4| 42|Female| 21.5|122.33|275.79| 1.19|63.64|0.54|-0.69|184.72|      49.36|
|  5| 47|  Male|31.62|137.18|232.35|-1.65|36.68|0.26| 1.63| 99.83|      54.15|
|  6| 22|  Male|37.06|106.47|244.34|-0.07|34.22|0.48|-1.72| 65.41|      39.55|
|  7| 23|  Male|26.75|129.39|177.69|-0.37|42.03|0.68| 0.82|180.71|      33.12|
|  8| 44|Female|28.38|125.23|276.99| 0.69|55.18| 1.1|-0.52|134.71|      53.84|
|  9| 36|  Male|36.83| 121.9|174.81|  1.1|53.55|0.25

In [ ]:
df = df.repartition(4)
df.rdd.getNumPartitions()

4

# StringIndexer

In [ ]:
# Map a string column of labels (categorical column) to a column of label indices
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol = 'sex', outputCol = 'gender')

df = indexer.fit(df).transform(df)

df.show(10)

+-------+---+------+-----+------+------+-----+-----+----+-----+------+-----------+------+
|     id|age|   sex|  bmi|    bp|    tc|  ldl|  hdl| tch|  ltg|   glu|progression|gender|
+-------+---+------+-----+------+------+-----+-----+----+-----+------+-----------+------+
| 472488| 26|Female|25.32|114.54|168.25|-1.61|68.13|0.37| -0.3|128.61|       34.5|   0.0|
|1520320| 38|  Male|31.73|123.05|240.81| 1.23|46.34|0.62| 0.04| 90.12|      50.53|   1.0|
| 162973| 38|Female|36.36|131.61|194.66|-0.63|34.62|0.56| 0.76|  7.79|       50.5|   0.0|
| 282089| 40|  Male|41.27|138.13|225.29| 1.65|55.23|0.54| 1.65|180.08|      56.29|   1.0|
|1148029| 38|Female|34.23|112.83|183.88|-0.73|56.61|0.28| 2.24| 29.01|      47.99|   0.0|
|1029802| 36|  Male|35.37|123.56|105.21| 0.12|47.17|0.71| 0.27|120.77|      49.35|   1.0|
| 297053| 36|  Male|17.84|131.14|197.88| -1.8|57.55|0.64|  0.7|140.36|      38.41|   1.0|
|1516214| 61|  Male|24.54|134.69|111.84| 1.83|40.21|0.23| 1.17|140.94|      66.34|   1.0|
| 307847| 

# VectorAssembler

In [ ]:
# combine features into a single vector column

assembler = VectorAssembler(inputCols = ["age", "gender", "bmi","bp","tc","ldl","hdl","tch","ltg","glu"],
                            outputCol = 'features')

data = assembler.transform(df)

data = data.select('features', 'progression')
data.show(5, truncate = False)

+----------------------------------------------------------+-----------+
|features                                                  |progression|
+----------------------------------------------------------+-----------+
|[36.0,0.0,28.64,114.25,197.01,1.03,53.38,0.31,0.2,102.59] |46.9       |
|[6.0,0.0,28.93,126.79,323.62,-1.68,55.41,0.43,1.49,177.94]|17.92      |
|[47.0,0.0,40.98,103.17,181.03,0.55,55.59,0.57,1.42,118.99]|60.83      |
|[36.0,1.0,22.73,135.4,191.81,-1.23,47.01,0.58,-0.66,98.06]|42.56      |
|[35.0,0.0,19.07,109.96,108.78,1.83,49.29,0.66,0.68,27.85] |41.6       |
+----------------------------------------------------------+-----------+
only showing top 5 rows



# StandardScaler

In [ ]:
scaler = StandardScaler(inputCol = 'features', outputCol = 'scaledFeatures')
#scaler = MinMaxScaler(inputCol = 'features', outputCol = 'scaledFeatures')

scaler_model = scaler.fit(data) # compute the mean & std from data and store the necessary scaling parameters.
data = scaler_model.transform(data) # scale the features of rhe sata using parameteres learned during fit() step

data = data.select('scaledFeatures', 'progression')
data.show(5, truncate = False)

In [ ]:
# split data
train_data, test_data = data.randomSplit([0.7, 0.3], seed = 42)
train_data.show(5, truncate = False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|scaledFeatures                                                                                                                                                                                  |progression|
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|[-0.9996801735890526,1.9999996870586678,4.51357334739171,14.255325909415204,4.91118826462631,-0.2996835328410389,3.6984618661746054,3.5515460876643856,0.6896706106762753,2.2420661268879787]   |7.79       |
|[-0.39987206943562104,1.9999996870586678,4.354876280061815,11.432865394690605,4.196960885559702,0.42954639707215575,4.967620359461673,1.2005226211823277,1.259398506452329,

# **Step 3:** Apply linear regression model.

In [ ]:
lr = LinearRegression(labelCol = 'progression', featuresCol = 'scaledFeatures', predictionCol = 'prediction')

# Fit the model to the training data
lr_model = lr.fit(train_data) # collect Beta0, Beta1, .....---> Y = B0 + B1X1

# Make prediction on the test data
lr_predictions = lr_model.transform(test_data)

In [ ]:
lr_predictions.select('prediction', 'progression').show(10, truncate = False)

+------------------+-----------+
|prediction        |progression|
+------------------+-----------+
|13.669604254407327|13.67      |
|16.15131807208173 |16.15      |
|15.454027752036483|15.45      |
|19.188021280832654|19.19      |
|10.159527245547876|10.16      |
|10.848031572992078|10.85      |
|10.1888271287878  |10.19      |
|13.806222966965121|13.81      |
|17.56023057023495 |17.56      |
|16.29993145172839 |16.3       |
+------------------+-----------+
only showing top 10 rows



In [ ]:
# Access the coefficients and intecept of the model
coefficients = lr_model.coefficients
intercept = lr_model.intercept

print(coefficients)
print(intercept)

# Predictor (Regressor): Progression =  2.9654446075967323e-05 + 8.502720630777699 AGE + -3.790026394372246e-06 GENDER + ................

[8.502720630777656,-3.790026382106173e-06,3.497225751266471,0.0999868488267554,-1.6300405615377437e-06,1.0010564848260575,-1.6886029128158215e-06,-0.019995443462289073,-0.8003843190067241,-2.911407426385012e-07]
2.9654446075968008e-05


# **Step 4:** Apply Lasso and Ridge regression models.
For Lasso regression, you would set elasticNetParam to 1.0, indicating that you want to use L1 regularization. For Ridge regression, you would set elasticNetParam to 0.0, indicating that you want to use L2 regularization.


In [ ]:
# Laaso
lasso = LinearRegression(labelCol = 'progression', featuresCol = 'scaledFeatures',
                         predictionCol = 'prediction', elasticNetParam = 1.0, regParam = 0.15) # lambda: tuning
lasso_model = lasso.fit(train_data)
lasso_predictions = lasso_model.transform(test_data)

# Ridge
ridge = LinearRegression(labelCol = 'progression', featuresCol = 'scaledFeatures',
                         predictionCol = 'prediction', elasticNetParam = 0.0, regParam = 0.15) # lambda: tuning
ridge_model = ridge.fit(train_data)
ridge_predictions = ridge_model.transform(test_data)

# **Step 5:** Evaluate the models and visualize the results.

In [ ]:
evaluator_mse = RegressionEvaluator(labelCol = 'progression', predictionCol = 'prediction', metricName = 'mse')
# calculate MSE
mse1 = evaluator_mse.evaluate(lr_predictions)
mse2 = evaluator_mse.evaluate(lasso_predictions)
mse3 = evaluator_mse.evaluate(ridge_predictions)

evaluator_rmse = RegressionEvaluator(labelCol = 'progression', predictionCol = 'prediction', metricName = 'rmse')
# calculate RMSE
rmse1 = evaluator_rmse.evaluate(lr_predictions)
rmse2 = evaluator_rmse.evaluate(lasso_predictions)
rmse3 = evaluator_rmse.evaluate(ridge_predictions)

evaluator_r2 = RegressionEvaluator(labelCol = 'progression', predictionCol = 'prediction', metricName = 'r2')
# calculate R_squared
r2_score1 = evaluator_r2.evaluate(lr_predictions)
r2_score2 = evaluator_r2.evaluate(lasso_predictions)
r2_score3 = evaluator_r2.evaluate(ridge_predictions)